Extraemos archivo jsonl que contiene la data cruda

#**🧪 PRUEBA TÉCNICA – INGENIERO DE DATOS**
**Empresa**: PUNTORED  
**Objetivo**: Transformaciones requeridas en la en la capa Silver.  
**Fecha creación**: 29/03/2025  
**Autor**: Jorge Andres Mosquera  
**Version**: 1.0   
**Fecha modificación**:  
**Motivo modificación**:  

Configuración de variables del notebook

In [0]:
# Configuración
catalog_name = "jorge_catalog"

bronze_table   = f"{catalog_name}.transacciones.bronze"
users_table    = f"{catalog_name}.silver.users"
details_table  = f"{catalog_name}.silver.transaction_details"
transactions_table = f"{catalog_name}.silver.transactions"

# Seleccionar catálogo y crear esquemas
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql("CREATE SCHEMA IF NOT EXISTS transacciones")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# Lectura desde Bronze como tabla UC
df_bronze = spark.table(bronze_table)

# --- Transformaciones ---
from pyspark.sql.functions import col, lower

df_clean = df_bronze.dropDuplicates(["id"]) \
    .filter(col("amount_in_cents") > 0) \
    .withColumn("status", lower(col("status"))) \
    .filter(col("created_at").isNotNull())

# Users
df_users = df_clean.select(
    col("payment_method_type.extra.card_holder").alias("user_id"),
    col("payment_method_type.type").alias("name"),
    col("created_at")
)
df_users.write.format("delta").mode("overwrite").saveAsTable(users_table)

# Details
df_details = df_clean.select(
    col("id").alias("transaction_id"),
    col("payment_method_type.installments").alias("installments"),
    col("payment_method_type.extra.processor_response_code").alias("response_code"),
    col("updated_at")
)
df_details.write.format("delta").mode("overwrite").saveAsTable(details_table)

# Transactions con MERGE incremental
df_transactions = df_clean.select(
    col("id").alias("transaction_id"),
    col("payment_method_type.extra.card_holder").alias("user_id"),
    col("amount_in_cents").alias("amount"),
    col("status"),
    col("created_at")
)

# Crear tabla si no existe
spark.sql(f"CREATE TABLE IF NOT EXISTS {transactions_table} (transaction_id STRING, user_id STRING, amount INT, status STRING, created_at TIMESTAMP) USING DELTA")

# MERGE
df_transactions.createOrReplaceTempView("updates")

spark.sql(f"""
MERGE INTO {transactions_table} AS target
USING updates AS source
ON target.transaction_id = source.transaction_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

# Validación
print("Users:", spark.table(users_table).count())
print("Details:", spark.table(details_table).count())
print("Transactions:", spark.table(transactions_table).count())

Users: 50000
Details: 50000
Transactions: 50000


Lectura datos desde capa bronze

Limpieza y reglas de negocio

In [0]:
from pyspark.sql.functions import col, lower

df_clean = df_bronze.dropDuplicates(["id"]) \
    .filter(col("amount_in_cents") > 0) \
    .withColumn("status", lower(col("status"))) \
    .filter(col("created_at").isNotNull())

In [0]:
print("Transactions:", spark.read.format("delta").load(silver_path + "/transactions").count())
print("Users:", spark.read.format("delta").load(silver_path + "/users").count())
print("Details:", spark.read.format("delta").load(silver_path + "/transaction_details").count())

Transactions: 50000
Users: 40
Details: 50000
